In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# CLIP: TRAINING + EVALUATION

!pip install -q transformers datasets torch scikit-learn pillow

In [4]:
# ---- IMPORTS ----
import torch, numpy as np
from transformers import CLIPProcessor, CLIPModel
from sklearn.metrics import accuracy_score, f1_score
from torch.nn.functional import normalize

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# ---- HF LOGIN ----
from huggingface_hub import login
login(token="your_token")

In [13]:
# ---- LOAD DATASET ----
from datasets import load_dataset
ds = load_dataset("Navarasa-9/navarasa_9")

README.md:   0%|          | 0.00/2.04k [00:00<?, ?B/s]

data/train-00000-of-00007.parquet:   0%|          | 0.00/480M [00:00<?, ?B/s]

data/train-00001-of-00007.parquet:   0%|          | 0.00/503M [00:00<?, ?B/s]

data/train-00002-of-00007.parquet:   0%|          | 0.00/500M [00:00<?, ?B/s]

data/train-00003-of-00007.parquet:   0%|          | 0.00/502M [00:00<?, ?B/s]

...

data/train-00005-of-00007.parquet:   0%|          | 0.00/493M [00:00<?, ?B/s]

data/train-00006-of-00007.parquet:   0%|          | 0.00/488M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/387M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12532 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1393 [00:00<?, ? examples/s]

In [14]:
# columns: image, navarasa, text
labels = sorted(set(ds["train"]["navarasa"]))
label2id = {l: i for i, l in enumerate(labels)}

ds = ds.map(lambda x: {"label": label2id[x["navarasa"]]})

Map:   0%|          | 0/12532 [00:00<?, ? examples/s]

Map:   0%|          | 0/1393 [00:00<?, ? examples/s]

In [15]:
# ---- METRICS ----
def recall_at_k(sim, k):
    return sum(i in sim[i].topk(k).indices for i in range(sim.size(0))) / sim.size(0)

def retrieval_metrics(img_emb, txt_emb):
    sim = img_emb @ txt_emb.T
    return {
        "R@1": recall_at_k(sim, 1),
        "R@5": recall_at_k(sim, 5),
        "R@10": recall_at_k(sim, 10),
    }

# ---- MODEL ----
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

In [16]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    images = [x["image"] for x in batch]
    texts  = [x["text"] for x in batch]
    labels = [x["label"] for x in batch]

    return images, texts, torch.tensor(labels)

In [17]:
train_loader = DataLoader(
    ds["train"],
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn
)

In [18]:
# ---- MODEL ----
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

In [19]:
# ----TRAIN ----
model.train()

for epoch in range(10):
    epoch_loss = 0

    for images, texts, _ in train_loader:
        inputs = processor(
            images=images,
            text=texts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        outputs = model(**inputs, return_loss=True)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()

    print(f"[CLIP] Epoch {epoch+1} | Avg Loss: {epoch_loss / len(train_loader):.4f}")


KeyboardInterrupt: 

In [20]:
!pip install -q tqdm

In [21]:
from tqdm import tqdm

In [22]:
num_epochs = 10

model.train()
for epoch in range(num_epochs):
    epoch_loss = 0.0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{num_epochs}",
        leave=True
    )

    for step, (images, texts, _) in enumerate(progress_bar):
        inputs = processor(
            images=images,
            text=texts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        outputs = model(**inputs, return_loss=True)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()

        # 🔹 update tqdm bar
        progress_bar.set_postfix({
            "step": step,
            "loss": f"{loss.item():.4f}"
        })

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Avg Loss: {epoch_loss / len(train_loader):.4f}"
    )

Epoch 1/10: 100%|██████████| 784/784 [08:23<00:00,  1.56it/s, step=783, loss=0.0012]


Epoch [1/10] Avg Loss: 0.3508


Epoch 2/10: 100%|██████████| 784/784 [08:23<00:00,  1.56it/s, step=783, loss=0.0026]


Epoch [2/10] Avg Loss: 0.2501


Epoch 3/10: 100%|██████████| 784/784 [08:21<00:00,  1.56it/s, step=783, loss=0.0018]


...

Epoch [8/10] Avg Loss: 0.1674


Epoch 9/10: 100%|██████████| 784/784 [08:20<00:00,  1.57it/s, step=783, loss=0.0703]


Epoch [9/10] Avg Loss: 0.1646


Epoch 10/10: 100%|██████████| 784/784 [08:26<00:00,  1.55it/s, step=783, loss=0.0001]

Epoch [10/10] Avg Loss: 0.1567


In [23]:
clip_save_dir = "/content/drive/MyDrive/models/clip-navarasa"

model.save_pretrained(clip_save_dir)
processor.save_pretrained(clip_save_dir)

print("CLIP model saved to Google Drive at:", clip_save_dir)

CLIP model saved to Google Drive at: /content/drive/MyDrive/models/clip-navarasa


In [24]:
!zip -r /content/clip-navarasa.zip /content/drive/MyDrive/models/clip-navarasa

  adding: content/drive/MyDrive/models/clip-navarasa/ (stored 0%)
  adding: content/drive/MyDrive/models/clip-navarasa/config.json (deflated 66%)
  adding: content/drive/MyDrive/models/clip-navarasa/model.safetensors (deflated 7%)
  adding: content/drive/MyDrive/models/clip-navarasa/preprocessor_config.json (deflated 50%)
  adding: content/drive/MyDrive/models/clip-navarasa/tokenizer_config.json (deflated 63%)
  adding: content/drive/MyDrive/models/clip-navarasa/special_tokens_map.json (deflated 78%)
  adding: content/drive/MyDrive/models/clip-navarasa/vocab.json (deflated 62%)
  adding: content/drive/MyDrive/models/clip-navarasa/merges.txt (deflated 60%)
  adding: content/drive/MyDrive/models/clip-navarasa/tokenizer.json (deflated 83%)


In [25]:
from google.colab import files

files.download("/content/clip-navarasa.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
# ---- EVALUATE ----
model.eval()
img_emb, txt_emb, y_true, y_pred = [], [], [], []

with torch.no_grad():
    for s in ds["test"]:
        img = processor(images=s["image"], return_tensors="pt").to(device)
        txt = processor(text=s["text"], return_tensors="pt", padding=True).to(device)

        i = normalize(model.get_image_features(**img), dim=-1)
        t = normalize(model.get_text_features(**txt), dim=-1)

        img_emb.append(i)
        txt_emb.append(t)

        y_pred.append((i @ t.T).argmax().item())
        y_true.append(s["label"])

img_emb = torch.cat(img_emb)
txt_emb = torch.cat(txt_emb)

print("\n[CLIP Evaluation]")
print("Retrieval:", retrieval_metrics(img_emb, txt_emb))
print("Accuracy:", accuracy_score(y_true, y_pred))
print("F1 Macro:", f1_score(y_true, y_pred, average="macro"))
print("F1 Weighted:", f1_score(y_true, y_pred, average="weighted"))



[CLIP Evaluation]
Retrieval: {'R@1': 0.12921751615218952, 'R@5': 0.4824120603015075, 'R@10': 0.7207465900933238}
Accuracy: 0.05671213208901651
F1 Macro: 0.013417119565217392
F1 Weighted: 0.006087307656293893
